# EEG-GNN FYP — run everything on Google Colab
Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom. Results are saved to your Google Drive so a
disconnect never loses work (`--resume` continues where it stopped).

In [ ]:
# Clone the portfolio repository and enter the project directory.
import os
REPO_URL = "https://github.com/krithika-bot/KRITHIKA.git"
REPO_DIR = "/content/KRITHIKA"
PROJECT_DIR = os.path.join(REPO_DIR, "projects", "eeg-gnn-fyp")
if not os.path.isdir(PROJECT_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $PROJECT_DIR
!pip -q install -e ".[test]"
!nvidia-smi | head -15


In [ ]:
# Keep quick and full results in separate Drive folders; this preserves prior runs.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
RESULTS = Path('/content/drive/MyDrive/eeg-fyp-results')
QUICK = RESULTS / 'quick'
FULL = RESULTS / 'full'
QUICK.mkdir(parents=True, exist_ok=True)
FULL.mkdir(parents=True, exist_ok=True)
print('Quick-run output:', QUICK)
print('Full-run output:', FULL)


## 3) Tests — all must PASS (this is the first time the torch code runs, so read any error carefully)

In [ ]:
!python tests/run_tests.py

In [ ]:
# 4) 1-2 minute wiring check on synthetic data (NOT a result)
!python scripts/run_deep.py --synthetic --smoke --out /tmp/smoke

## 5) Real data — quick run first (20 subjects, 1 seed). Downloads from PhysioNet through MNE (a few minutes).

In [ ]:
N = 20
!python scripts/make_data_figures.py --n-subjects $N --out $QUICK/figures
!python scripts/run_baselines.py --n-subjects $N --seeds 0 --out $QUICK/classical
!python scripts/run_baselines.py --n-subjects $N --seeds 0 --out $QUICK/classical --permute-train-labels


In [ ]:
!python scripts/run_deep.py --n-subjects $N --seeds 0 --out $QUICK/deep --resume


In [ ]:
!python scripts/make_summary.py --results $QUICK
from IPython.display import Image, display, Markdown
display(Markdown((QUICK / 'summary.md').read_text()))
for f in ['erd_sanity', 'method_comparison', 'confusion_matrices', 'learning_curves']:
    path = QUICK / 'figures' / f'{f}.png'
    if path.exists():
        display(Image(str(path)))


## Full experiment (all eligible subjects, three seeds)

Run this only after reviewing the quick run. It writes to the separate `full/` Drive folder and does not delete or overwrite `quick/`. Deep-model progress can be resumed with `--resume`.


In [ ]:
!python scripts/make_data_figures.py --out $FULL/figures
!python scripts/run_baselines.py --out $FULL/classical
!python scripts/run_baselines.py --out $FULL/classical --permute-train-labels
!python scripts/run_deep.py --out $FULL/deep --resume
!python scripts/make_summary.py --results $FULL


## Save reviewed evidence to GitHub

After a real-data run completes and the summary is reviewed, copy its `summary.md`, `summary.csv`, `comparisons.csv`, figures, prediction CSVs, and `run_info.json` files into this project’s `results/` folder. Do not commit synthetic smoke outputs as scientific results or redistribute raw PhysioNet EEG files. Update the README and portfolio snippet only with claims supported by the committed run artifacts.
